# Task 2 — Data Discovery, Profiling and Cleaning

Profiles and cleans the freeCodeCamp dataset produced by `01_extract.ipynb`, and saves the result to `data/interim/cleaned.csv`.

## Imports

In [1]:
import pandas as pd
import glob

## Load the data

Loads whichever `freecodecamp_*.csv` is in `data/raw/` — no need to hardcode the date.

In [2]:
raw_files = glob.glob("../data/raw/freecodecamp_*.csv")
latest_file = sorted(raw_files)[-1]
print(f"Loading: {latest_file}")

df = pd.read_csv(latest_file)
print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
df.head()

Loading: ../data/raw\freecodecamp_2026-09-14.csv
Loaded 450 rows, 9 columns


,source,category,title,author,publication_date,description,url,topic,matched_keywords
0,freeCodeCamp,#shadcn ui,How to Build an AI Chat App Interface With the...,Vaibhav Gupta,2026-09-11T16:21:29.864Z,Every other AI product you open today has the ...,https://www.freecodecamp.org/news/how-to-build...,AI,NaN
1,freeCodeCamp,#Artificial Intelligence,How to Build a Self-Evaluating AI System: Auto...,Jude Otine,2026-09-11T15:24:04.941Z,So you shipped your AI feature and it works in...,https://www.freecodecamp.org/news/build-a-self...,AI,"artificial intelligence, llm"
2,freeCodeCamp,#Security,How AI Is Changing Malware Detection: From Tra...,Manish Shivanandhan,2026-09-11T15:22:46.931Z,Malware used to be simple to describe. A virus...,https://www.freecodecamp.org/news/how-ai-is-ch...,AI,NaN
3,freeCodeCamp,#AI,How to Build an AI Chatbot with Gemini and Ver...,Johnson Samuel,2026-09-07T22:35:39.060Z,"A couple of months back, I built a chatbot app...",https://www.freecodecamp.org/news/how-to-build...,AI,NaN
4,freeCodeCamp,#AI,How AI Receptionists Work: The Architecture Be...,Manish Shivanandhan,2026-09-04T20:07:46.216Z,An AI receptionist may sound simple from the o...,https://www.freecodecamp.org/news/how-ai-recep...,AI,NaN


## Profiling

One row per column: dtype, null count, percent null, unique count, sample value.

In [3]:
profile_rows = []
for col in df.columns:
    profile_rows.append({
        "column": col,
        "dtype": str(df[col].dtype),
        "null_count": df[col].isnull().sum(),
        "percent_null": round(df[col].isnull().mean() * 100, 1),
        "unique_count": df[col].nunique(),
        "sample_value": df[col].dropna().iloc[0] if df[col].notna().any() else None
    })

profile_df = pd.DataFrame(profile_rows)
profile_df

,column,dtype,null_count,percent_null,unique_count,sample_value
0,source,object,0,0.0,1,freeCodeCamp
1,category,object,0,0.0,157,#shadcn ui
2,title,object,0,0.0,449,How to Build an AI Chat App Interface With the...
3,author,object,0,0.0,153,Vaibhav Gupta
4,publication_date,object,0,0.0,450,2026-09-11T16:21:29.864Z
5,description,object,0,0.0,450,Every other AI product you open today has the ...
6,url,object,0,0.0,450,https://www.freecodecamp.org/news/how-to-build...
7,topic,object,0,0.0,3,AI
8,matched_keywords,object,198,44.0,76,"artificial intelligence, llm"


## Additional checks

In [4]:
print(f"Duplicate rows (full row): {df.duplicated().sum()}")
print(f"Duplicate URLs: {df.duplicated(subset=['url']).sum()}")
print(f"\nTopic distribution:\n{df['topic'].value_counts()}")
print(f"\n'No author' count: {(df['author'] == 'No author').sum()}")
print(f"'No date' count: {(df['publication_date'] == 'No date').sum()}")
print(f"'No description' count: {(df['description'] == 'No description').sum()}")

Duplicate rows (full row): 0
Duplicate URLs: 0

Topic distribution:
topic
AI              150
Cloud           150
Data Science    150
Name: count, dtype: int64

'No author' count: 0
'No date' count: 0
'No description' count: 0


## Issues found and decisions

| Issue | Decision |
|---|---|
| `matched_keywords` mostly empty (NaN) | Kept as-is — it's an enrichment field, not a required field. Filled with `"none"` for consistency instead of leaving NaN. |
| `publication_date` in ISO 8601 with time (`...T16:21:29.864Z`) | Converted to a plain date column for readability; kept the full timestamp in a separate column in case time-of-day is needed later. |
| `category` (site's own tag, e.g. `#shadcn ui`) doesn't always match `topic` (our AI/Cloud/Data Science label) | Kept both columns — they answer different questions (the site's own tagging vs. our classification). Not a data quality issue, just two different labels. |
| Duplicate URLs | Checked and removed if found (see below). |
| `author` / `description` occasionally `"No author"` / `"No description"` | Left as an explicit placeholder rather than blank/NaN, so it's clear the value was looked for and not found (vs. never being fetched). |

## Clean the data

In [5]:
df_clean = df.copy()

# Drop exact duplicate rows and duplicate URLs
df_clean = df_clean.drop_duplicates()
df_clean = df_clean.drop_duplicates(subset=["url"])

# Standardise dates: keep both a full timestamp and a simple date
df_clean["publication_datetime"] = pd.to_datetime(df_clean["publication_date"], errors="coerce", utc=True)
df_clean["publication_date"] = df_clean["publication_datetime"].dt.date

# Fill the enrichment column consistently instead of leaving NaN
df_clean["matched_keywords"] = df_clean["matched_keywords"].fillna("none")

# Column names are already snake_case, nothing to rename here

print(f"Rows before cleaning: {len(df)}")
print(f"Rows after cleaning: {len(df_clean)}")
df_clean.head()

Rows before cleaning: 450
Rows after cleaning: 450


,source,category,title,author,publication_date,description,url,topic,matched_keywords,publication_datetime
0,freeCodeCamp,#shadcn ui,How to Build an AI Chat App Interface With the...,Vaibhav Gupta,2026-09-11,Every other AI product you open today has the ...,https://www.freecodecamp.org/news/how-to-build...,AI,none,2026-09-11 16:21:29.864000+00:00
1,freeCodeCamp,#Artificial Intelligence,How to Build a Self-Evaluating AI System: Auto...,Jude Otine,2026-09-11,So you shipped your AI feature and it works in...,https://www.freecodecamp.org/news/build-a-self...,AI,"artificial intelligence, llm",2026-09-11 15:24:04.941000+00:00
2,freeCodeCamp,#Security,How AI Is Changing Malware Detection: From Tra...,Manish Shivanandhan,2026-09-11,Malware used to be simple to describe. A virus...,https://www.freecodecamp.org/news/how-ai-is-ch...,AI,none,2026-09-11 15:22:46.931000+00:00
3,freeCodeCamp,#AI,How to Build an AI Chatbot with Gemini and Ver...,Johnson Samuel,2026-09-07,"A couple of months back, I built a chatbot app...",https://www.freecodecamp.org/news/how-to-build...,AI,none,2026-09-07 22:35:39.060000+00:00
4,freeCodeCamp,#AI,How AI Receptionists Work: The Architecture Be...,Manish Shivanandhan,2026-09-04,An AI receptionist may sound simple from the o...,https://www.freecodecamp.org/news/how-ai-recep...,AI,none,2026-09-04 20:07:46.216000+00:00


## Save to `data/interim/`

In [6]:
output_file = "../data/interim/cleaned.csv"
df_clean.to_csv(output_file, index=False)
print(f"Saved {len(df_clean)} rows to {output_file}")

Saved 450 rows to ../data/interim/cleaned.csv
